# Proyecto Integrador 2 · Parte III — Recomendador colaborativo
## Tareas 17 y 18: vecindario (item–item) y ALS en Spark MLlib

**Pregunta de interés P2:** *¿Qué proveedores son sustitutos entre sí para una misma entidad o categoría, y qué afinidades entidad–proveedor son inusuales?*

**Idea del recomendador colaborativo:** no mira qué venden los proveedores, solo **quién contrata a quién**. Si muchas entidades que contratan al proveedor A también contratan al B, entonces A y B "van juntos" y se le recomienda B a la entidad que solo tiene A.

Se prueban dos formas:
1. **Vecindario:** similitud directa. Se prueban las dos formas vistas en clase: **item–item** (entre proveedores) y **user–user** (entre entidades). Fácil de explicar.
2. **ALS (factorización de matrices, Spark MLlib):** resume cada entidad y cada proveedor en `r` números (factores). Si los factores de una entidad y de un proveedor "encajan", se recomienda.

**Entrada:** `interacciones_train` del notebook 1. **Salida:** `rec_vecindario` (item–item), `rec_vecindario_user`, `rec_als` (Parquet), los factores de ALS, y las tablas para la web (`vecindario_k.csv`, `vecindario_user_k.csv`, `vecindario_variantes.csv`, `als_parametros.csv`, `als_alpha.csv`).

**Importante:** los parámetros (cuántos vecinos, cuántos factores) se eligen con una **validación dentro de train** (el 20% de pares más recientes). El **test** (desde 2026-03-01) no se toca aquí; se usa recién en el notebook 6.

## 0. Configuración

In [1]:
import os, json, time
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

RUTA_P2 = os.environ.get("P2_DATOS", "/content/drive/MyDrive/proyecto_integrador_seace/p2")
WEB     = os.environ.get("P2_WEB",   "/content/drive/MyDrive/proyecto_integrador_seace/web_artefactos/p2")
os.makedirs(WEB, exist_ok=True)

K_REC     = 50                        # recomendaciones guardadas por entidad
K_EVAL    = 10                        # las métricas se miden en el top-10
VECINOS   = [5, 10, 25, 50, 100, 200, 400, 800, 1600] # vecindario item–item: cuántos proveedores vecinos se prueban
VECINOS_U = [5, 10, 25, 50, 100, 200, 400, 800]       # vecindario user–user: cuántas entidades vecinas se prueban
RANGOS    = [8, 16, 32, 64, 128, 256]    # factores de ALS que se prueban
REGS      = [0.01, 0.1, 1.0, 10.0]    # regularización de ALS que se prueba
ALPHAS    = [1.0, 10.0, 40.0]         # confianza de ALS en cada interacción observada
PCT_VAL   = 0.80                      # los pares con fecha_primera sobre este percentil van a validación
SEED      = 42

## 1. Matriz entidad × proveedor y partición de validación
- `R[e, p] = 1` si la entidad `e` le adjudicó al menos un proceso al proveedor `p` (antes del corte).
- **Validación:** se ordenan los pares por `fecha_primera` y el 20% más reciente es el "futuro" que se intenta adivinar, usando solo el 80% anterior. Es la misma idea del test, un poco más atrás en el tiempo.

In [2]:
train = pd.read_parquet(f"{RUTA_P2}/interacciones_train")
ent_ids  = np.sort(train.entidad_idx.unique()); prov_ids = np.sort(train.proveedor_idx.unique())
pos_ent  = {e: i for i, e in enumerate(ent_ids)}; pos_prov = {p: i for i, p in enumerate(prov_ids)}
train["e"] = train.entidad_idx.map(pos_ent); train["p"] = train.proveedor_idx.map(pos_prov)
NE, NP = len(ent_ids), len(prov_ids)

def matriz(df):
    return csr_matrix((np.ones(len(df), dtype="float32"), (df.e.values, df.p.values)), shape=(NE, NP))

corte_val = train.fecha_primera.quantile(PCT_VAL)
tr_in = train[train.fecha_primera <= corte_val]      # lo que se usa para entrenar al elegir parámetros
va    = train[train.fecha_primera >  corte_val]      # lo que se intenta adivinar
print("pares en train:", len(train), "| entrenar (80%):", len(tr_in), "| validar (20%):", len(va), "| corte:", str(corte_val)[:10])

# En validación solo se puntúan pares cuyo proveedor y entidad ya existían en el 80% de entrenamiento
ok = va.e.isin(tr_in.e) & va.p.isin(tr_in.p)
print(f"pares de validación evaluables: {ok.sum()} de {len(va)} ({100*ok.mean():.0f}%)")
va = va[ok]

pares en train: 44205 | entrenar (80%): 35366 | validar (20%): 8839 | corte: 2025-08-05
pares de validación evaluables: 7820 de 8839 (88%)


## 2. Cómo se mide: Recall@10
Para cada entidad se ordenan todos los proveedores que **todavía no tiene** y se mira el top-10.
- **Recall@10:** de los proveedores nuevos que la entidad sí contrató después, ¿qué fracción quedó en el top-10? (0 = ninguno, 1 = todos).

In [3]:
def metricas(S, R_vistos, df_val, k=K_EVAL):
    """S: puntajes (entidades × proveedores). R_vistos: lo que la entidad ya tiene (se excluye). df_val: pares a adivinar."""
    S = S.copy(); S[R_vistos.nonzero()] = -np.inf
    rel = df_val.groupby("e").p.apply(set)
    ents = rel.index.values
    top = np.argpartition(-S[ents], k, axis=1)[:, :k]
    orden = np.argsort(-np.take_along_axis(S[ents], top, axis=1), axis=1)
    top = np.take_along_axis(top, orden, axis=1)
    rec = []
    for i, e in enumerate(ents):
        r = rel[e]; aciertos = np.array([p in r for p in top[i]])
        rec.append(aciertos.sum() / len(r))
    return float(np.mean(rec)), len(ents)

R_in = matriz(tr_in)
# Referencia: recomendar los proveedores más populares (con más entidades)
pop = np.asarray(R_in.sum(0)).ravel()
base = metricas(np.tile(pop, (NE, 1)).astype("float32"), R_in, va)
print(f"referencia 'más populares': recall@10 = {base[0]:.4f} | entidades evaluadas: {base[1]}")

referencia 'más populares': recall@10 = 0.0278 | entidades evaluadas: 1602


## 3. Vecindario item–item
1. **Similitud coseno entre proveedores:** `sim(A,B) = (entidades en común) / √(entidades de A · entidades de B)`. Va de 0 (nunca coinciden) a 1 (siempre las mismas entidades).
2. Por cada proveedor se guardan solo sus **K vecinos** más similares.
3. **Puntaje** de un proveedor `q` para una entidad: suma de `sim(p, q)` sobre los proveedores `p` que la entidad ya tiene y que tienen a `q` como vecino.

Se prueban varios `K`: pocos vecinos = recomendaciones muy parecidas a lo ya contratado; muchos = se mezcla más ruido.

In [4]:
def similitud(R):
    co = (R.T @ R).toarray().astype("float32")           # entidades en común entre cada par de proveedores
    grado = np.sqrt(np.diag(co)); grado[grado == 0] = 1
    S = co / grado[:, None] / grado[None, :]
    np.fill_diagonal(S, 0)
    return S

def solo_k_vecinos(S, k):
    S2 = np.zeros_like(S); idx = np.argpartition(-S, k, axis=1)[:, :k]
    np.put_along_axis(S2, idx, np.take_along_axis(S, idx, axis=1), axis=1)
    return S2

S_in = similitud(R_in)
filas = []
for k in VECINOS:
    puntaje = R_in @ solo_k_vecinos(S_in, k)
    r, ne = metricas(np.asarray(puntaje), R_in, va)
    filas.append({"vecinos": k, "recall@10": round(r, 4)})
    print(f"K={k:>3}  recall@10={r:.4f}")
vec_k = pd.DataFrame(filas)
vec_k["referencia_recall@10"] = round(base[0], 4)
vec_k.to_csv(f"{WEB}/vecindario_k.csv", index=False)
K_BEST = int(vec_k.sort_values("recall@10", ascending=False).vecinos.iloc[0])
print("\nK elegido (mejor recall@10):", K_BEST)

K=  5  recall@10=0.0585


K= 10  recall@10=0.0583


K= 25  recall@10=0.0622


K= 50  recall@10=0.0613


K=100  recall@10=0.0614


K=200  recall@10=0.0662


K=400  recall@10=0.0690


K=800  recall@10=0.0732


K=1600  recall@10=0.0721

K elegido (mejor recall@10): 800


## 3b. Vecindario user–user (entre entidades) y comparación de similitudes
En clase se vieron dos formas de vecindario y tres similitudes. Aquí se prueban con los mismos datos, para decidir con números y no por costumbre:
- **User–user:** en vez de comparar proveedores, se comparan **entidades** (coseno entre las listas de proveedores que contrata cada una). El puntaje de un proveedor para una entidad es la suma de la similitud con las entidades vecinas que lo contratan. Cada entidad guarda sus **K entidades vecinas**.
- **Coseno:** entidades en común ÷ √(entidades de A · entidades de B). Es la que usamos en item–item.
- **Jaccard:** entidades en común ÷ entidades de A o de B. Castiga más a los proveedores muy grandes.
- **Pearson (centrada):** como los valores son 0/1, es la correlación entre dos columnas de ceros y unos (coeficiente *phi*). Centra respecto al promedio de cada proveedor.

Con calificaciones explícitas (1 a 5 estrellas) Pearson corrige que cada persona califica distinto. Aquí no hay calificaciones, solo "existe / no existe", por eso se comprueba en vez de suponer.

In [5]:
S_ent = similitud(R_in.T.tocsr())                       # entidad × entidad (coseno)
filas_u = []
for k in VECINOS_U:
    r, _ = metricas(np.asarray(solo_k_vecinos(S_ent, k) @ R_in), R_in, va)
    filas_u.append({"vecinos": k, "recall@10": round(r, 4)})
    print(f"user–user K={k:>3}  recall@10={r:.4f}")
vec_u = pd.DataFrame(filas_u)
vec_u["referencia_recall@10"] = round(base[0], 4)
vec_u.to_csv(f"{WEB}/vecindario_user_k.csv", index=False)
K_BEST_U = int(vec_u.sort_values("recall@10", ascending=False).vecinos.iloc[0])
print("\nK elegido user–user (mejor recall@10):", K_BEST_U)

def sim_jaccard(R):
    co = (R.T @ R).toarray().astype("float32"); g = np.diag(co).copy()
    S = co / (g[:, None] + g[None, :] - co + 1e-9); np.fill_diagonal(S, 0); return S

def sim_pearson(R):
    co = (R.T @ R).toarray().astype("float32"); g = np.diag(co).copy(); N = R.shape[0]
    num = N * co - g[:, None] * g[None, :]
    den = np.sqrt(g[:, None] * (N - g[:, None]) * g[None, :] * (N - g[None, :])) + 1e-9
    S = num / den; np.fill_diagonal(S, 0); return np.clip(S, 0, None)   # solo se usan similitudes positivas

filas_v = []
for forma, M, Kb, comb in [("item–item", R_in, K_BEST, lambda S, k: R_in @ solo_k_vecinos(S, k)),
                           ("user–user", R_in.T.tocsr(), K_BEST_U, lambda S, k: solo_k_vecinos(S, k) @ R_in)]:
    for nombre, f in [("coseno", similitud), ("Jaccard", sim_jaccard), ("Pearson", sim_pearson)]:
        r, _ = metricas(np.asarray(comb(f(M), Kb)), R_in, va)
        filas_v.append({"forma": forma, "similitud": nombre, "vecinos": Kb, "recall@10": round(r, 4)})
vecindario_variantes = pd.DataFrame(filas_v); vecindario_variantes.to_csv(f"{WEB}/vecindario_variantes.csv", index=False)
print(vecindario_variantes.to_string(index=False))

user–user K=  5  recall@10=0.0770


user–user K= 10  recall@10=0.1039


user–user K= 25  recall@10=0.1166


user–user K= 50  recall@10=0.1246


user–user K=100  recall@10=0.1227


user–user K=200  recall@10=0.1210


user–user K=400  recall@10=0.1204


user–user K=800  recall@10=0.1204

K elegido user–user (mejor recall@10): 50


    forma similitud  vecinos  recall@10
item–item    coseno      800     0.0732
item–item   Jaccard      800     0.0729
item–item   Pearson      800     0.0686
user–user    coseno       50     0.1246
user–user   Jaccard       50     0.1240
user–user   Pearson       50     0.1237


## 4. ALS en Spark MLlib
**ALS** (Alternating Least Squares) busca para cada entidad un vector de `r` factores y para cada proveedor otro de `r` factores, de modo que el **producto punto** sea alto cuando hay relación. No usa calificaciones sino **interacciones implícitas** (`implicitPrefs=True`): que exista el par significa interés; que no exista no significa rechazo, solo que no se sabe.

Parámetros:
- `rank` (r): cuántos factores. Más factores = más detalle, pero más riesgo de memorizar.
- `regParam`: castigo a factores grandes. Más alto = modelo más simple.
- `alpha`: cuánta confianza da cada interacción observada (se prueban 1, 10 y 40; 10 es el valor usual para implícito).
- Valor de cada par: `log(1 + n_adjudicaciones)`, para que un proveedor con 200 contratos no pese 200 veces más que uno con 1.

In [6]:
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS

spark = (SparkSession.builder.master("local[2]").appName("P2_ALS")
         .config("spark.driver.memory", "3g").config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

def entrenar_als(df, rank, reg, alpha=10.0, iteraciones=10):
    sdf = spark.createDataFrame(pd.DataFrame({"e": df.e.astype(int), "p": df.p.astype(int),
                                              "valor": np.log1p(df.n_adjudicaciones).astype("float64")}))
    modelo = ALS(userCol="e", itemCol="p", ratingCol="valor", implicitPrefs=True, alpha=alpha,
                 rank=rank, regParam=reg, maxIter=iteraciones, seed=SEED, nonnegative=False).fit(sdf)
    U = np.zeros((NE, rank), dtype="float32"); I = np.zeros((NP, rank), dtype="float32")
    for r in modelo.userFactors.collect(): U[r.id] = r.features
    for r in modelo.itemFactors.collect(): I[r.id] = r.features
    return U, I

filas = []
for rank in RANGOS:
    for reg in REGS:
        t0 = time.perf_counter()
        U, I = entrenar_als(tr_in, rank, reg)
        r, _ = metricas(U @ I.T, R_in, va)
        filas.append({"factores": rank, "regularizacion": reg, "recall@10": round(r, 4)})
        print(f"rank={rank:>2} reg={reg:<5} recall@10={r:.4f}  ({time.perf_counter()-t0:.0f} s)")
als_p = pd.DataFrame(filas)
als_p["referencia_recall@10"] = round(base[0], 4)
als_p.to_csv(f"{WEB}/als_parametros.csv", index=False)
mejor = als_p.sort_values("recall@10", ascending=False).iloc[0]
RANK_BEST, REG_BEST = int(mejor.factores), float(mejor.regularizacion)
print("\nMejor combinación (factores, regParam):", RANK_BEST, REG_BEST)
# Con esa combinación se prueba también alpha (la confianza que se da a cada interacción observada)
filas_a = []
for a in ALPHAS:
    U, I = entrenar_als(tr_in, RANK_BEST, REG_BEST, alpha=a)
    r, _ = metricas(U @ I.T, R_in, va)
    filas_a.append({"alpha": a, "recall@10": round(r, 4)})
    print(f"alpha={a:<5} recall@10={r:.4f}")
als_alpha = pd.DataFrame(filas_a)
als_alpha.to_csv(f"{WEB}/als_alpha.csv", index=False)
ALPHA_BEST = float(als_alpha.sort_values("recall@10", ascending=False).alpha.iloc[0])
print("\nALS elegido:", RANK_BEST, "factores, regParam", REG_BEST, ", alpha", ALPHA_BEST)
print("¿Algún parámetro quedó en el borde de su rejilla?  factores:", RANK_BEST in (min(RANGOS), max(RANGOS)),
      "| regParam:", REG_BEST in (min(REGS), max(REGS)), "| alpha:", ALPHA_BEST in (min(ALPHAS), max(ALPHAS)),
      "| vecinos:", K_BEST in (min(VECINOS), max(VECINOS)))

rank= 8 reg=0.01  recall@10=0.0768  (13 s)


rank= 8 reg=0.1   recall@10=0.0765  (6 s)


rank= 8 reg=1.0   recall@10=0.0680  (5 s)


rank= 8 reg=10.0  recall@10=0.0013  (5 s)


rank=16 reg=0.01  recall@10=0.1039  (5 s)


rank=16 reg=0.1   recall@10=0.1029  (5 s)


... (17 outputs left out)

rank=256 reg=10.0  recall@10=0.0065  (139 s)

Mejor combinación (factores, regParam): 128 1.0


alpha=1.0   recall@10=0.0988


alpha=10.0  recall@10=0.1332


alpha=40.0  recall@10=0.1238

ALS elegido: 128 factores, regParam 1.0 , alpha 10.0
¿Algún parámetro quedó en el borde de su rejilla?  factores: False | regParam: False | alpha: False | vecinos: False


## 5. Reentrenar con todo train y guardar recomendaciones
Con los parámetros elegidos se entrena de nuevo usando **todo train** (ya sin separar validación) y se guardan las 50 mejores recomendaciones por entidad (sin los proveedores que ya tiene).

In [7]:
R_full = matriz(train)

def top_recomendaciones(S, nombre):
    S = np.asarray(S, dtype="float32").copy(); S[R_full.nonzero()] = -np.inf
    top = np.argpartition(-S, K_REC, axis=1)[:, :K_REC]
    sc = np.take_along_axis(S, top, axis=1); o = np.argsort(-sc, axis=1)
    top = np.take_along_axis(top, o, axis=1); sc = np.take_along_axis(sc, o, axis=1)
    rec = pd.DataFrame({"entidad_idx": np.repeat(ent_ids, K_REC), "proveedor_idx": prov_ids[top.ravel()],
                        "score": sc.ravel().astype(float), "rank": np.tile(np.arange(1, K_REC + 1), NE)})
    rec = rec[np.isfinite(rec.score) & (rec.score > 0)]            # sin puntaje positivo no hay motivo para recomendar
    rec.to_parquet(f"{RUTA_P2}/{nombre}.parquet", index=False)
    print(nombre, ":", len(rec), "filas |", rec.entidad_idx.nunique(), "entidades | proveedores distintos:", rec.proveedor_idx.nunique())

S_full = similitud(R_full)
top_recomendaciones(R_full @ solo_k_vecinos(S_full, K_BEST), "rec_vecindario")
top_recomendaciones(solo_k_vecinos(similitud(R_full.T.tocsr()), K_BEST_U) @ R_full, "rec_vecindario_user")

U, I = entrenar_als(train, RANK_BEST, REG_BEST, alpha=ALPHA_BEST)
np.save(f"{RUTA_P2}/als_factores_entidad.npy", U); np.save(f"{RUTA_P2}/als_factores_proveedor.npy", I)
top_recomendaciones(U @ I.T, "rec_als")
spark.stop()

rec_vecindario : 114171 filas | 2331 entidades | proveedores distintos: 9388


rec_vecindario_user : 114171 filas | 2331 entidades | proveedores distintos: 7135


rec_als : 116550 filas | 2331 entidades | proveedores distintos: 8422


## 6. Resumen de la elección

In [8]:
print(f"Referencia 'más populares'   : recall@10 = {base[0]:.4f}")
v = vec_k.set_index('vecinos').loc[K_BEST]; print(f"Vecindario item–item (K={K_BEST}) : recall@10 = {v['recall@10']:.4f}")
u = vec_u.set_index('vecinos').loc[K_BEST_U]; print(f"Vecindario user–user (K={K_BEST_U})   : recall@10 = {u['recall@10']:.4f}")
a = als_p.sort_values('recall@10', ascending=False).iloc[0]; print(f"ALS ({RANK_BEST} factores, reg {REG_BEST}) : recall@10 = {a['recall@10']:.4f}")
print("\n(todo medido en la validación dentro de train; el resultado final sobre el test está en el notebook 6)")

Referencia 'más populares'   : recall@10 = 0.0278
Vecindario item–item (K=800) : recall@10 = 0.0732
Vecindario user–user (K=50)   : recall@10 = 0.1246
ALS (128 factores, reg 1.0) : recall@10 = 0.1332

(todo medido en la validación dentro de train; el resultado final sobre el test está en el notebook 6)
